In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path, PureWindowsPath

device = (torch.device("mps") if torch.backends.mps.is_available() else
          torch.device("cuda") if torch.cuda.is_available() else
          torch.device("cpu"))
print("Device:", device)

PATCH_SIZE   = 224
patches_root = Path("/kaggle/input/datasets/rohinipa1cb/patches-for-her2-cancer/patches")
model_dir    = Path("/kaggle/input/datasets/rohinipa1cb/her2-cancer-saved-models14thnov/25fold_results_23rdNov_models")
val_out = Path("val_output")
val_out.mkdir(parents=True, exist_ok=True)

# ── patch index with paths rebuilt for this OS ───────────────────────
patch_index = pd.read_csv(patches_root / "patch_index.csv")
patch_index["spot_id"] = patch_index["spot_id"].astype(str)
patch_index["patient"] = patch_index["patient"].astype(int)
patch_index["mmap_path"] = patch_index["mmap_path"].apply(
    lambda p: str(patches_root / PureWindowsPath(p).name))
FILE_COUNTS = patch_index.groupby("mmap_path").size().to_dict()

# ── data loading ─────────────────────────────────────────────────────
def load_spots(path, her2_map):
    df = pd.read_csv(path, index_col=0)
    df.index.name = "spot_id"
    df = df.reset_index()
    df["spot_id"] = df["spot_id"].astype(str)
    df["patient"] = df["patient"].astype(int)
    df["label"]   = df["patient"].map(her2_map).map({"HER2-": 0, "HER2+": 1})
    assert df["label"].notna().all(), f"{path}: patient without HER2 label"
    return df

class HER2Dataset(Dataset):
    def __init__(self, spots, mean, std):
        m = spots.merge(patch_index[["spot_id", "patient", "section",
                                     "mmap_path", "mmap_idx"]],
                        on=["spot_id", "patient", "section"], how="inner")
        assert len(m) == len(spots), f"{len(spots) - len(m)} spots have no patch"

        self.meta   = m[["spot_id", "patient", "section", "label"]].reset_index(drop=True)
        self.paths  = m["mmap_path"].values
        self.idx    = m["mmap_idx"].astype(int).values
        self.labels = m["label"].values.astype(np.float32)
        self.mmaps  = {p: np.memmap(p, dtype="uint8", mode="r",
                                    shape=(FILE_COUNTS[p], PATCH_SIZE, PATCH_SIZE, 3))
                       for p in np.unique(self.paths)}
        self.mean = torch.tensor(mean).view(3, 1, 1)
        self.std  = torch.tensor(std).view(3, 1, 1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        patch = np.array(self.mmaps[self.paths[i]][self.idx[i]])
        x = torch.from_numpy(patch).permute(2, 0, 1).float() / 255.0
        x = (x - self.mean) / self.std
        return x, torch.tensor(self.labels[i])

# ── model ────────────────────────────────────────────────────────────
def conv_block(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
        nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
        nn.MaxPool2d(2))

class HER2CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(conv_block(3, 32), conv_block(32, 64),
                                     conv_block(64, 128), conv_block(128, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(256, 1))

    def forward(self, x):
        return self.head(self.encoder(x))

# ── validation data ──────────────────────────────────────────────────
val_pats     = pd.read_csv("/kaggle/input/datasets/rohinipa1cb/val-patients-list-her2cancer/val_patients.csv")
her2_map_val = dict(zip(val_pats["patient"].astype(int), val_pats["her2"]))
val_spots    = load_spots(
    "/kaggle/input/datasets/rohinipa1cb/val-spots-tumor-her2cancer/val_spots_tumor.csv",
    her2_map_val)
print(f"Validation spots: {len(val_spots)} | patients: {val_spots['patient'].nunique()}")

# ── validate each model ──────────────────────────────────────────────
def validate_model(model_path):
    ck    = torch.load(model_path, map_location=device)
    model = HER2CNN().to(device)
    model.load_state_dict(ck["state_dict"])
    model.eval()

    # normalise with the statistics this model was trained with
    ds = HER2Dataset(val_spots, mean=ck["mean"], std=ck["std"])

    logits, labels = [], []
    with torch.no_grad():
        for x, y in DataLoader(ds, batch_size=64, shuffle=False):
            logits.append(model(x.to(device)).squeeze(1).cpu())
            labels.append(y)
    logits = torch.cat(logits)
    labels = torch.cat(labels)

    val_loss = nn.functional.binary_cross_entropy_with_logits(logits, labels).item()
    probs    = torch.sigmoid(logits).numpy()

    name  = model_path.stem                      # e.g. "model_1"
    preds = pd.DataFrame({
        "model":        name,
        "set":          "val",
        "spot_id":      ds.meta["spot_id"].values,
        "patient":      ds.meta["patient"].values,
        "section":      ds.meta["section"].values,
        "true_label":   ds.labels.astype(int),
        "pred_label":   (probs >= 0.5).astype(int),
        "prob_HER2pos": probs,
    })
    preds.to_csv(val_out / f"{name}_predictions_val.csv", index=False)
    return dict(model=name, val_loss=round(val_loss, 5), n_val_spots=len(ds))

rows = []
for i in range(1, 26):
    p = model_dir / f"model_{i}.pt"
    if not p.exists():
        print(f"model_{i}.pt: not found, skipping")
        continue
    row = validate_model(p)
    rows.append(row)
    print(f"model_{i} | val loss {row['val_loss']:.4f} | spots {row['n_val_spots']}")

pd.DataFrame(rows).to_csv(val_out / "val_loss_summary.csv", index=False)
print("\nSaved per-model predictions and val_loss_summary.csv to", val_out)